In [1]:
import os

# --- Fix TF 2.18 + protobuf 6 crash (MessageFactory.GetPrototype) ---
# Must be set before importing tensorflow (or anything that triggers protobuf C++ runtime).
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION_VERSION"] = "2"
os.environ["TF_USE_PYTHON_PROTOBUF"] = "1"

# Pre-import protobuf api_implementation to ensure the python implementation is selected early.
# Avoid calling private _SetType hooks (can be unstable under protobuf 6).
from google.protobuf.internal import api_implementation  # noqa: F401

import glob
import random
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from tqdm.notebook import tqdm
except Exception:
    from tqdm import tqdm

import pydicom  # Handle MRI images
import cv2  # OpenCV

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.utils import to_categorical

print("TF:", tf.__version__)
print("pydicom:", pydicom.__version__)



2026-01-23 02:32:50.865416: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769135570.878869      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769135570.882999      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-23 02:32:50.901557: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

TF: 2.18.0
pydicom: 3.0.1


In [2]:
data_dir = Path("../input/rsna-miccai-brain-tumor-radiogenomic-classification/")

mri_types = ["FLAIR", "T1w", "T2w", "T1wCE"]
excluded_images = [109, 123, 709]  # Bad images (per competition note)



In [3]:
train_df = pd.read_csv(data_dir / "train_labels.csv")
test_df = pd.read_csv(data_dir / "sample_submission.csv")
sample_submission = pd.read_csv(data_dir / "sample_submission.csv")

train_df["BraTS21ID"] = train_df["BraTS21ID"].astype(int)
test_df["BraTS21ID"] = test_df["BraTS21ID"].astype(int)
sample_submission["BraTS21ID"] = sample_submission["BraTS21ID"].astype(int)

train_df = train_df[~train_df.BraTS21ID.isin(excluded_images)].reset_index(drop=True)
print(f"train data: Rows={train_df.shape[0]}, Columns={train_df.shape[1]}")
train_df.head()




train data: Rows=523, Columns=2


,BraTS21ID,MGMT_value
0,642,0
1,100,1
2,309,0
3,301,0
4,464,0


In [4]:
def load_dicom(path, size=388):
    """
    Reads a DICOM image, standardizes so that the pixel values are between 0 and 1,
    then rescales to 0..255 and resizes.
    """
    dicom = pydicom.dcmread(path)
    data = dicom.pixel_array.astype(np.float32)

    mx = float(np.max(data)) if data.size else 0.0
    if mx != 0.0:
        data = data / mx
    data = (data * 255.0).astype(np.uint8)

    return cv2.resize(data, (size, size), interpolation=cv2.INTER_AREA)




In [5]:
def _dicom_sort_key(p):
    stem = os.path.splitext(os.path.basename(p))[0]
    try:
        return int(stem.split("-")[-1])
    except Exception:
        return stem


def get_all_image_paths(brats21id, image_type, folder="train"):
    """
    Returns an array of paths for a particular sequence type for one subject.
    """
    assert image_type in mri_types

    patient_path = os.path.join(
        str(data_dir / folder),
        str(int(brats21id)).zfill(5),
    )

    paths = sorted(
        glob.glob(os.path.join(patient_path, image_type, "*")),
        key=_dicom_sort_key,
    )

    num_images = len(paths)
    if num_images == 0:
        return np.array([])

    start = int(num_images * 0.25)
    end = int(num_images * 0.75)

    interval = 3
    if num_images < 10:
        interval = 1

    return np.array(paths[start:end:interval])


def get_all_images(brats21id, image_type, folder="train", size=225):
    paths = get_all_image_paths(brats21id, image_type, folder)
    return [load_dicom(path, size) for path in paths]




In [6]:
def get_all_data_for_train(image_type, image_size=32):
    global train_df

    X = []
    y = []
    train_ids = []

    for i in tqdm(train_df.index):
        x = train_df.loc[i]
        images = get_all_images(int(x["BraTS21ID"]), image_type, "train", image_size)
        if len(images) == 0:
            continue

        label = int(x["MGMT_value"])
        X += images
        y += [label] * len(images)
        train_ids += [int(x["BraTS21ID"])] * len(images)

    return np.array(X), np.array(y), np.array(train_ids)


def get_all_data_for_test(image_type, image_size=32):
    global test_df

    X = []
    test_ids = []

    for i in tqdm(test_df.index):
        x = test_df.loc[i]
        images = get_all_images(int(x["BraTS21ID"]), image_type, "test", image_size)
        if len(images) == 0:
            continue
        X += images
        test_ids += [int(x["BraTS21ID"])] * len(images)

    return np.array(X), np.array(test_ids)




In [7]:
X, y, trainidt = get_all_data_for_train("T1wCE", image_size=32)
X_test, testidt = get_all_data_for_test("T1wCE", image_size=32)

print("X:", X.shape, "y:", y.shape, "trainidt:", trainidt.shape)
print("X_test:", X_test.shape, "testidt:", testidt.shape)



  0%|          | 0/523 [00:00<?, ?it/s]

  0%|          | 0/59 [00:00<?, ?it/s]

X: (14563, 32, 32) y: (14563,) trainidt: (14563,)
X_test: (1633, 32, 32) testidt: (1633,)


In [8]:
X_train, X_valid, y_train, y_valid, trainidt_train, trainidt_valid = train_test_split(
    X, y, trainidt, random_state=15, stratify=y
)

X_train = tf.expand_dims(X_train, axis=-1)
X_valid = tf.expand_dims(X_valid, axis=-1)

if X_test.size == 0:
    X_test = tf.zeros((0,) + tuple(X_train.shape[1:]), dtype=X_train.dtype)
else:
    X_test = tf.expand_dims(X_test, axis=-1)

y_train = to_categorical(y_train, num_classes=2)
y_valid = to_categorical(y_valid, num_classes=2)

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_valid:", X_valid.shape, "y_valid:", y_valid.shape)




X_train: (10922, 32, 32, 1) y_train: (10922, 2)
X_valid: (3641, 32, 32, 1) y_valid: (3641, 2)


2026-01-23 02:33:15.171677: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1769135595.172122      11 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:05:00.0, compute capability: 8.6


In [9]:
def get_model01(width=128, height=128, depth=64, name="3dcnn"):
    """Build a 3D convolutional neural network model."""
    inputs = tf.keras.Input((width, height, depth, 1))

    x = tf.keras.layers.Conv3D(filters=64, kernel_size=3, activation="relu")(inputs)
    x = tf.keras.layers.MaxPool3D(pool_size=2)(x)
    x = tf.keras.layers.BatchNormalization()(x)

    x = tf.keras.layers.Conv3D(filters=64, kernel_size=3, activation="relu")(x)
    x = tf.keras.layers.MaxPool3D(pool_size=2)(x)
    x = tf.keras.layers.BatchNormalization()(x)

    x = tf.keras.layers.Conv3D(filters=128, kernel_size=3, activation="relu")(x)
    x = tf.keras.layers.MaxPool3D(pool_size=2)(x)
    x = tf.keras.layers.BatchNormalization()(x)

    x = tf.keras.layers.Conv3D(filters=256, kernel_size=3, activation="relu")(x)
    x = tf.keras.layers.MaxPool3D(pool_size=2)(x)
    x = tf.keras.layers.BatchNormalization()(x)

    x = tf.keras.layers.GlobalAveragePooling3D()(x)
    x = tf.keras.layers.Dense(units=512, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)

    outputs = tf.keras.layers.Dense(units=1, activation="sigmoid")(x)

    model = tf.keras.Model(inputs, outputs, name=name)

    initial_learning_rate = 0.00009
    lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate, decay_steps=100000, decay_rate=0.96, staircase=True
    )
    model.compile(
        loss="binary_crossentropy",
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
        metrics=["acc"],
    )

    return model




In [10]:
def get_model02():
    np.random.seed(0)
    random.seed(12)
    tf.random.set_seed(12)

    inpt = keras.Input(shape=X_train.shape[1:])

    h = keras.layers.Rescaling(1.0 / 255.0)(inpt)

    h = keras.layers.Conv2D(64, kernel_size=(4, 4), activation="relu", name="Conv_1")(h)
    h = keras.layers.MaxPool2D(pool_size=(2, 2))(h)

    h = keras.layers.Conv2D(32, kernel_size=(2, 2), activation="relu", name="Conv_2")(h)
    h = keras.layers.MaxPool2D(pool_size=(1, 1))(h)

    h = keras.layers.Dropout(0.1)(h)

    h = keras.layers.Flatten()(h)
    h = keras.layers.Dense(32, activation="relu")(h)

    output = keras.layers.Dense(2, activation="softmax")(h)

    model = keras.Model(inpt, output)

    model.compile(
        loss="categorical_crossentropy",
        optimizer="adam",
        metrics=[tf.keras.metrics.AUC(name="auc")],
    )
    return model




In [11]:
checkpoint_filepath = "best_model.keras"

model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_filepath,
    save_weights_only=False,
    monitor="val_auc",
    mode="max",
    save_best_only=True,
    save_freq="epoch",
    verbose=1,
)



In [12]:
model = get_model02()
history = model.fit(
    x=X_train,
    y=y_train,
    epochs=25,
    callbacks=[model_checkpoint_callback],
    validation_data=(X_valid, y_valid),
    verbose=2,
)



Epoch 1/25


I0000 00:00:1769135596.924013      68 service.cc:148] XLA service 0x7f017c00bec0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1769135596.924051      68 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2026-01-23 02:33:16.947041: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1769135597.049404      68 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1769135598.369143      68 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.



Epoch 1: val_auc improved from -inf to 0.62243, saving model to best_model.keras
342/342 - 5s - 15ms/step - auc: 0.5895 - loss: 0.6814 - val_auc: 0.6224 - val_loss: 0.6737
Epoch 2/25

Epoch 2: val_auc improved from 0.62243 to 0.63626, saving model to best_model.keras
342/342 - 1s - 2ms/step - auc: 0.6263 - loss: 0.6679 - val_auc: 0.6363 - val_loss: 0.6676
Epoch 3/25

Epoch 3: val_auc improved from 0.63626 to 0.64836, saving model to best_model.keras
342/342 - 1s - 2ms/step - auc: 0.6608 - loss: 0.6511 - val_auc: 0.6484 - val_loss: 0.6560
Epoch 4/25

Epoch 4: val_auc improved from 0.64836 to 0.68148, saving model to best_model.keras
342/342 - 1s - 2ms/step - auc: 0.7002 - loss: 0.6258 - val_auc: 0.6815 - val_loss: 0.6367
Epoch 5/25

Epoch 5: val_auc improved from 0.68148 to 0.71458, saving model to best_model.keras
342/342 - 1s - 2ms/step - auc: 0.7432 - loss: 0.5943 - val_auc: 0.7146 - val_loss: 0.6163
Epoch 6/25

Epoch 6: val_auc improved from 0.71458 to 0.73975, saving model to best

In [13]:
model_best = None
if os.path.exists(checkpoint_filepath):
    try:
        model_best = tf.keras.models.load_model(filepath=checkpoint_filepath)
    except Exception as e:
        print(
            "WARNING: load_model failed, using in-memory model instead. Error:", repr(e)
        )
        model_best = model
else:
    print(
        "WARNING: checkpoint not found, using in-memory model instead:",
        checkpoint_filepath,
    )
    model_best = model

y_pred_valid = model_best.predict(X_valid, verbose=0)[:, 1]

valid_df = pd.DataFrame(
    {"BraTS21ID": trainidt_valid.astype(int), "MGMT_value_pred": y_pred_valid}
)
valid_df = valid_df.groupby("BraTS21ID", as_index=False).mean()
valid_df = valid_df.merge(
    train_df[["BraTS21ID", "MGMT_value"]], on="BraTS21ID", how="inner"
)

auc = roc_auc_score(valid_df["MGMT_value"], valid_df["MGMT_value_pred"])
print(f"Validation AUC={auc:.5f} on {len(valid_df)} subjects")



Validation AUC=0.91203 on 508 subjects


In [14]:
if int(X_test.shape[0]) == 0:
    y_pred_test = np.array([], dtype=np.float32)
else:
    y_pred_test = model_best.predict(X_test, verbose=0)[:, 1]

# Safety: ensure dataframe has the expected columns even if empty
test_pred_df = pd.DataFrame(
    {"BraTS21ID": testidt.astype(int), "MGMT_value": y_pred_test}
)
if len(test_pred_df) > 0:
    test_pred_df = test_pred_df.groupby("BraTS21ID", as_index=False).mean()
else:
    test_pred_df = pd.DataFrame(
        {"BraTS21ID": np.array([], dtype=int), "MGMT_value": np.array([], dtype=float)}
    )

sub = sample_submission[["BraTS21ID"]].copy()
sub = sub.merge(test_pred_df, on="BraTS21ID", how="left")

sub["MGMT_value"] = sub["MGMT_value"].fillna(0.5).astype(float)
sub["BraTS21ID"] = sub["BraTS21ID"].astype(int).astype(str).str.zfill(5)

sub = sub[["BraTS21ID", "MGMT_value"]]

sub.to_csv("submission.csv", index=False)
print("Wrote submission.csv with shape:", sub.shape)
sub.head()

Wrote submission.csv with shape: (59, 2)


,BraTS21ID,MGMT_value
0,00356,0.455895
1,00140,0.751147
2,00757,0.702774
3,00275,0.721946
4,00570,0.643135
